# `LocalPermutation`

`LocalPermutation` shuffles the rows of an n-site dataset so that each
row moves **at most** `threshold` distance units — a spatially constrained
permutation.  Unlike `LocalBootstrap`, each row appears **exactly once**
(no replacement).  Optionally enforced as a **derangement**: no row may
stay at its original site.

### Algorithm

1. **Build adjacency** — A[i,j] = True iff site j is within `threshold`
   of site i (diagonal blocked for derangements).  A `libpysal.graph.Graph`
   can be used instead, optionally combined with `threshold` to filter
   edges by weight.

2. **Initial permutation — Hungarian algorithm** — solve the *linear
   assignment problem*: find a complete matching between sites and values
   using only feasible pairs.  Random costs in [0, 1] are assigned to
   feasible pairs for random tie-breaking, then
   `scipy.optimize.linear_sum_assignment` solves in O(n³).

3. **Markov chain mixing** — propose swapping `perm[i]` and `perm[j]`;
   accept iff both moves stay within A and no fixed point is created.
   Run `n_burn` steps (default `10 * n`) between each yielded permutation.

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geodatasets
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from geovalidate import LocalPermutation

%matplotlib inline

## Simple example — Chicago community areas

With 77 community areas we can clearly see how the derangement works.
`threshold = 8 km` gives every area at least 3 swap partners.

The right panel shows **one permuted realisation**: each area displays
the income of a nearby area that swapped with it.  No area keeps its own
value (derangement), and every value still appears exactly once.
Compare with `LocalBootstrap` where values can repeat.

In [ ]:
import geodatasets, numpy as np, geopandas as gpd

chicago = gpd.read_file(geodatasets.get_path("geoda.chicago_health")).to_crs("EPSG:32616")
chicago["income"] = chicago["PerCInc14"].fillna(chicago["PerCInc14"].median())
print(f"n = {len(chicago)} community areas")

In [ ]:
y = chicago["income"].values

lp_simple = LocalPermutation(threshold=8_000, derangement=True,
                             n_permutations=1, random_state=42)
perm_simple = next(lp_simple.sample(chicago))
y_perm_simple = y[perm_simple]

# Verify it is a true derangement
assert all(perm_simple[i] != i for i in range(len(chicago))), "Fixed point found!"
assert len(set(perm_simple)) == len(chicago), "Not a valid permutation!"
print("Valid derangement confirmed.")
print(f"Max displacement: {max(abs(perm_simple[i]-i) for i in range(len(chicago)))} index positions")

In [ ]:
norm = mcolors.Normalize(vmin=y.min(), vmax=y.max())
PT   = dict(norm=norm, cmap="RdYlGn", edgecolor="#333", linewidth=0.5, legend=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.patch.set_facecolor("#f8f8f8")

chicago_p = chicago.copy()

chicago_p["_v"] = y
chicago_p.plot(column="_v", ax=axes[0], **PT)
axes[0].set_title("Original\nPer-capita income (USD)", fontsize=11)
axes[0].set_aspect("equal"); axes[0].axis("off")

chicago_p["_v"] = y_perm_simple
chicago_p.plot(column="_v", ax=axes[1], **PT)
axes[1].set_title("One derangement realisation\n(no area keeps its own value)", fontsize=11)
axes[1].set_aspect("equal"); axes[1].axis("off")

sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
fig.colorbar(sm, ax=axes, label="Per-capita income (USD)", fraction=0.02, pad=0.02)
fig.suptitle("LocalPermutation — Chicago community areas  (threshold = 8 km, no replacement)", fontsize=12)
fig.tight_layout()
plt.show()

## Ensemble use — King County house sales

Applying `perm` to **entire rows** of [X, y] jointly creates a complete
locally-permuted training dataset.  Training a model on each gives a
*LocalPermutation ensemble* — analogous to `LocalBootstrap` but without
replacement.  High ensemble uncertainty marks areas where predictions are
sensitive to which nearby houses happen to be included in training.

In [ ]:
import geodatasets, numpy as np, geopandas as gpd

gdf_full = gpd.read_file(geodatasets.get_path("geoda.home_sales")).to_crs("EPSG:32610")
gdf_full["log_price"] = np.log(gdf_full["price"])
gdf_full["decile"] = (
    gdf_full["log_price"].rank(pct=True).multiply(10).clip(upper=9.99).astype(int)
)
idx = (
    gdf_full.groupby("decile")
    .apply(lambda g: g.sample(min(60, len(g)), random_state=42), include_groups=False)
    .index.get_level_values(1)
)
gdf = gdf_full.loc[idx].reset_index(drop=True)
print(f"n = {len(gdf)} sales  |  price ${gdf.price.min():,.0f} - ${gdf.price.max():,.0f}")

In [ ]:
FEATURES = ["sqft_liv", "bedrooms", "bathrooms", "grade",
            "floors", "waterfront", "view", "condition"]
X = gdf[FEATURES].fillna(0).values.astype(float)
y = gdf["log_price"].values

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X, y)
print(f"Baseline R2 = {1 - np.var(y - model.predict(X))/np.var(y):.3f}")

In [ ]:
THRESHOLD = 12_000
N_PERM    = 100

lp = LocalPermutation(threshold=THRESHOLD, derangement=True,
                      n_permutations=N_PERM, random_state=0)

ensemble_preds = np.empty((N_PERM, len(gdf)))
for k, perm in enumerate(lp.sample(gdf)):
    model.fit(X[perm], y[perm])
    ensemble_preds[k] = model.predict(X[perm])

ens_std = ensemble_preds.std(axis=0)
print(f"Ensemble std range: {ens_std.min():.3f} – {ens_std.max():.3f} (log USD)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("#f8f8f8")
PT2 = dict(s=16, alpha=0.85, edgecolors="white", linewidths=0.4)

ax = axes[0]
sc = ax.scatter(gdf.geometry.x/1000, gdf.geometry.y/1000,
                c=gdf["price"], cmap="YlOrRd",
                norm=mcolors.LogNorm(gdf.price.min(), gdf.price.max()), **PT2)
plt.colorbar(sc, ax=ax, label="Sale price (USD)", fraction=0.04, pad=0.02)
ax.set_xlabel("Easting (km)"); ax.set_ylabel("Northing (km)")
ax.set_title(f"King County sale prices\n(n={len(gdf)}, stratified subsample)", fontsize=11)
ax.set_aspect("equal")

ax2 = axes[1]
sc2 = ax2.scatter(gdf.geometry.x/1000, gdf.geometry.y/1000,
                  c=ens_std, cmap="PuRd",
                  norm=mcolors.Normalize(ens_std.min(), ens_std.max()), **PT2)
plt.colorbar(sc2, ax=ax2, label="Ensemble std dev (log price)", fraction=0.04, pad=0.02)
ax2.set_xlabel("Easting (km)"); ax2.set_ylabel("Northing (km)")
ax2.set_title(f"Ensemble uncertainty\n({N_PERM} derangements, threshold={THRESHOLD//1000} km)",
              fontsize=11)
ax2.set_aspect("equal")

fig.suptitle("LocalPermutation ensemble — King County (UTM 10N)", fontsize=13)
fig.tight_layout()
plt.show()